## Test training

Attention encoder-decoder

Let $f$ be a list of $n_f$ features for $m$ options and $c$ be a list of $n_c$ characteristics, which are 
non-market data features such as strike, time to expiry. The objective is to encode-decode the one of "price", "vol"
or "sqrtVar" for each option.

For the moment

```
    CHARACTERISTICS = {
        'tte_y',
        'nms',
        'strike'
    }
    FEATURES = CHARACTERISTICS | {
        'ask_pure_price',
        'bid_pure_price',
        'ask_vol',
        'bid_vol',
    }
```


### Encoding

Define:

* $V\equiv V( f ) \in \mathbb{R}^{m, n_h}$ be the value projection from $n_f$ features to $n_h$ heads per option.
* $Q\equiv Q( f ) \approx ( A_Q f - b_Q ) \in \mathbb{R}^{m, n_h, n_{qk}}$ be the query projection from $n_f$ features to $n_h$ queries/keys, with an inner dimension $n_{qk}$ per option.
* $K\equiv K( f ) \approx ( A_K f - b_K ) \in \mathbb{R}^{m, n_h, n_{qk}}$ be the key projection from $n_f$ features to $n_h$ queries/keys, with an inner dimension $n_{qk}$ per option.
    
Define the raw weights as a sort of Gaussian version of attention:
$$
     \alpha :=  \exp\left( -\frac1{\sqrt{n_{qk}} } \sum_{k=1}^{n_{qk}} Q_{o,j,k}  K_{o,j,k} \right) = \exp\left( -\frac1{\sqrt{n_{qk}} }  Q_{o,j}'  K_{o,j} \right)  \in \mathbb{R}^{m, n_h} \ .
$$
We then define "heads" for $j=1,\ldots,n_h$ as
$$
    h_j := \frac{ \sum_{o=1}^m \alpha_{o,j} V_{o,j} }{  \sum_{o=1}^m  \alpha_{o,j} }
$$
The hidden state is then computed across heads as
$$
    s= \mathcal{N}\left( h \right) \in \mathbb{R}^{n_s}
$$
where $\mathcal{N}$ is a final network.

### Decoding
We decode per option - that means we do not do attention accross options.

* $\bar V \equiv \bar V(c,s) \in \mathbb{R}^{m, \bar n_h}$ are $\bar n_h$ candidate predictions per option.
* $\bar Q\equiv Q( c ) \approx ( \bar A_Q c - \bar b_Q ) \in \mathbb{R}^{m, \bar n_h, \bar n_{qk}}$ be the query projection from $\bar n_c$ characteristics to $\bar n_h$ queries/keys, with an inner dimension $\bar n_{qk}$.
* $\bar K\equiv K( c ) \approx ( \bar A_K c - \bar b_K ) \in \mathbb{R}^{m, \bar n_h, \bar n_{qk}}$ be the key projection from $\bar n_c$ features to $\bar n_h$ queries/keys, with an inner dimension $\bar n_{qk}$.

Define the raw weights
$$
    \bar \alpha :=  \exp\left( - \frac1{\sqrt{\bar n_{qk}} } \sum_{k=1}^{\bar n_{qk}} \bar Q_{o,j,k}  \bar K_{o,j,k} \right) = 
    \exp\left( - \frac1{\sqrt{\bar n_{qk}} }  \bar Q_{o,j}'  \bar K_{o,j} \right)  \in \mathbb{R}^{m, \bar n_h}
$$
and then return the predictor as weighted value, not sums:
$$
    \bar h_j := \frac{ \bar \alpha_{o,j} \bar V_{o,j} }{ \sum_{j=1}^{\bar n_h} \bar \alpha_{o,j} } 
$$
which we then decode into a forecast for each option $o$ as
$$
     y_o := \bar{\mathcal{N}}\left( \bar h \right)
$$
where $\bar{\mathcal{N}}$ is a final network.



In [1]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.sampler
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.torch
%aimport quantdata.torch.bbtorch
%aimport quantdata.torch.ffw
%aimport quantdata.dysanos.simple
%aimport quantdata.dysanos.simple.sampler
%aimport quantdata.dysanos.simple.model
%aimport quantdata.dysanos.simple.trainer
%aimport quantdata.dysanos.simple.aec

from quantdata.dysanos.simple.aec import load_data
sampler = load_data(["SPX"])
print("/done")



00: Importing torch...done; this took 0.00429ms.
00: Generating samples for 1 tickers SPX with from 2015-01-01 to 2025-07-01... done; this took 0.37s.
00: Found the following number of samples per ticker: SPX: 2617.
00: Found date range 2015-01-02 to 2025-07-01 with 2617 total dates. First validation date for 25 steps per sample is 2023-05-01.
00: Using up to 1000 options per date.
01:   Ticker SPX: using 411 training samples (accross 2079 training dates), and 103 validation samples (accross 538 validation dates).
00: Created 411 training samples and 103 validation samples based on the date split at 2023-05-01. That implies 20% of all samples are validation samples.
00: Total time taken: 0.41s.
/done


## AEC

In [2]:
from quantdata.dysanos.simple.model import *
from quantdata.dysanos.simple.aec import *


aec_config = AEC.Config(
    # core pararmeters
    encode_what = "vol",
    fit_what    = "price",
    num_state   = 64,
    fit_average = True,

    # rarely touch these
    weight_by_spread = False,

    encoder = AttentionPoolingEncoderFactory(
        num_head = 64,
        num_QK = 8,
        V_config = FeedForward.Config(
            num_depth = 0, # linear
            num_width = 64, # head
            activation = 'Softplus',
        ),
        QK_config = FeedForward.Config(
            num_depth = 0,  # linear
            num_width = 16, # twice num_QK
            activation = 'ReLU', # idea is to be sparse...?
        ),
        head_config = FeedForward.Config(
            num_depth = 2,
            num_width = 64, # head
            activation = 'ReLU', # idea is to be sparse...?
        )
    ),
    decoder = GatingDecoderFactory(
        num_head = 16,
        num_QK = 4,        
        V_config = FeedForward.Config(
            num_depth = 2,
            num_width = 32, # twice head
            activation = 'Softplus',    
        ),
        QK_config = FeedForward.Config(
            num_depth = 0, # linear
            num_width = 4, # equal QK
            activation = 'ReLU', # idea is to be sparse...?
        ),
        head_config = FeedForward.Config(
            num_depth = 2,
            num_width = 32, # twice head
            activation = 'Softplus',
        )
    ),
)

run( ipynbname.name(), version="1.1", aec_config=aec_config, sampler=sampler, epochs = 1000, batch_per_epoch = 10, debug=False,
     trn_batch_size=64,
    val_batch_size=128 )


NameError: name 'AttentionPoolingEncoderFactory' is not defined